# ETL - Etapa 2: Tratamento
- Lendo a base bruta, padroniza nomes e tipos, trata nulos e duplicados.

- Entrada: data/raw/cotacoes_brutas.csv

- Saída: data/processed/cotacoes_limpas.csv

In [148]:
# Bibliotecas:
from pathlib import Path
import pandas as pd

# Pasta RAIZ do projeto:
raiz = Path.cwd().parent

# Local do arquivo CSV com os dados brutos:
caminho = raiz / "data" / "raw" / "cotacoes_brutas.csv"

# Leitura dos dados brutos:
df_raw = pd.read_csv(caminho, sep=",", decimal=".", encoding="utf-8")

df_raw.head()  # Visualização do inicio dos dados brutos

,Date,Adj Close,Close,High,Low,Open,Volume,nome_acao
0,2021-01-04,32.996513,37.310001,38.195000,37.250000,37.924999,7219800,WEGE3
1,2021-01-05,35.021744,39.599998,39.599998,37.000000,37.325001,16633600,WEGE3
2,2021-01-06,35.950352,40.650002,41.735001,39.064999,39.685001,21507400,WEGE3
3,2021-01-07,37.436138,42.330002,42.505001,40.919998,41.189999,15129600,WEGE3
4,2021-01-08,39.700172,44.889999,44.994999,42.424999,42.474998,17381000,WEGE3


In [141]:
# Informações dos sobres os dados
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 8940 entries, 0 to 8939
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       8940 non-null   str    
 1   Adj Close  8940 non-null   float64
 2   Close      8940 non-null   float64
 3   High       8940 non-null   float64
 4   Low        8940 non-null   float64
 5   Open       8940 non-null   float64
 6   Volume     8940 non-null   int64  
 7   nome_acao  8940 non-null   str    
dtypes: float64(5), int64(1), str(2)
memory usage: 690.6 KB


**Observação:**
- Total de linhas: 8940, índices de 0 ao 8939.
- Total de Colunas: 8.

Descrições das colunas:
| Coluna| Tipo lido pelo pandas | Descrição |
|---|---|---|
| Date | `str` | Data da cotação. |
| Adj Close | `float` | Preço de fechamento ajustado por dividendos e desdobramentos. |
| Close | `float` | Preço de fechamento. |
| High | `float` | Preço máximo. |
| Low | `float` | Preço mínimo. |
| Open | `float` | Preço de abertura. |
| Volume | `int` | Quantidade de ações negociadas do dia. |
| nome_acao | `str` | Nome do ativo. |

 - Apenas uma coluna com nome em português. `nome_acao`
 - A coluna `Date` não foi lida como datetime64.
 - Nenhum valor nulo encontrado.


In [142]:
# Renomeando as colunas para português
df = df_raw.rename(
    columns={
        "Date": "data",
        "Adj Close": "fechamento_ajustado",
        "Close": "fechamento",
        "High": "maxima",
        "Low": "minima",
        "Open": "abertura",
        "Volume": "volume",
    }
)

In [143]:
# Corrigindo a leitura da coluna (data) para datetime64
df["data"] = pd.to_datetime(df["data"])

In [144]:
# Verifica se a mesma ação aparece mais de uma vez na mesma data
duplicados = df.duplicated(subset=["data", "nome_acao"]).sum()
print(f"Linhas duplicadas: {duplicados}")

Linhas duplicadas: 0


In [145]:
# Informações descritivas
df.describe()

,data,fechamento_ajustado,fechamento,maxima,minima,abertura,volume
count,8940,8940.000000,8940.000000,8940.000000,8940.000000,8940.000000,8.940000e+03
mean,2022-07-02 03:38:24.966443,24.628770,31.794496,32.270303,31.339733,31.814402,2.320298e+07
min,2021-01-04 00:00:00,5.697405,8.370000,8.680000,8.210000,8.340000,0.000000e+00
25%,2021-10-01 00:00:00,14.077126,19.243923,19.532988,18.969999,19.268409,6.604489e+06
50%,2022-07-04 00:00:00,18.641513,24.660000,25.061746,24.290001,24.660097,1.463085e+07
75%,2023-03-30 00:00:00,31.308897,34.650002,35.139999,34.209999,34.639999,3.153068e+07
max,2023-12-28 00:00:00,71.032074,118.720001,120.449997,115.250000,119.800003,4.991948e+08
std,NaN,14.839196,20.056211,20.320016,19.794520,20.075714,2.559222e+07


**Observação:**
- Primeira data registrada e menor: 21/01/2021, pois nos dias 01 a 03/01/2021 não houve pregão.
- Ultima data registrada e maior: 2023/12/28, pois nos dias 29 a 31/12/2023 não houve pregão.
- A coluna `volume` aprestou um valor igual a zero (0.0)

In [149]:
# Verificando a existência de valores negativos e quais são zero
print(f"Valores <= 0 presentes em cada coluna:\n{(df[[
        "fechamento_ajustado",
        "fechamento",
        "maxima",
        "minima",
        "abertura",
        "volume"
    ]] <= 0).sum().to_string()}")

# Valore máximo do dia não pode ser menor que o mínimo
print(f"\nMáxima <  Mínima: {(df["maxima"] < df["minima"]).sum()}\n")

# Quais linhas tem o voluma igual a zero?
df[df["volume"] == 0]

Valores <= 0 presentes em cada coluna:
fechamento_ajustado    0
fechamento             0
maxima                 0
minima                 0
abertura               0
volume                 3

Máxima <  Mínima: 0



,data,fechamento_ajustado,fechamento,maxima,minima,abertura,volume,nome_acao
431,2022-09-26,27.793694,30.570000,30.570000,30.570000,30.570000,0,WEGE3
4901,2022-09-26,28.074907,28.120001,28.120001,28.120001,28.120001,0,PRIO3
7306,2023-06-02,11.721806,15.780000,15.780000,15.780000,15.780000,0,BBDC4


**Observação:**
- Volume zero:
    - 3 registros (WEGE3 e PRIO3 em 26/09/2022, BBDC4 em 02/06/2023) apresentam volume zero e preços de abertura, máxima, mínima e fechamento idênticos, um forte indicativo de um erro de registro, pois parece que não foi negociado nesse dia, o que seria improvavel.
- Irei manter eles pois não iram impactar na análise.